# 05. Advanced Analytics
This notebook covers advanced risk modeling, forecasting, portfolio optimization, and investor analytics.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from datetime import datetime, timedelta
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import scipy.optimize as sco
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../charts', exist_ok=True)


## 1. Data Loading

In [2]:
# Load main NAV dataset
df_nav = pd.read_csv('../data/processed/merged_dataset.csv')
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav.sort_values(['fund_id', 'date'], inplace=True)
df_nav['daily_return'] = df_nav.groupby('fund_id')['nav'].pct_change()
df_nav.dropna(subset=['daily_return'], inplace=True)

# Load investor data
df_inv = pd.read_csv('../data/processed/investor_transactions_clean.csv')
df_inv['transaction_date'] = pd.to_datetime(df_inv['transaction_date'], dayfirst=True)

print("Data loaded successfully.")


Data loaded successfully.


## 2. Historical VaR & Conditional VaR (CVaR)
Evaluating the 95% and 99% Value at Risk and Expected Shortfall for each fund.

In [3]:
var_cvar_results = []
for fund_id, group in df_nav.groupby('fund_id'):
    returns = group['daily_return'].dropna()
    if len(returns) == 0:
        continue
    
    var_95 = np.percentile(returns, 5)
    cvar_95 = returns[returns <= var_95].mean()
    
    var_99 = np.percentile(returns, 1)
    cvar_99 = returns[returns <= var_99].mean()
    
    var_cvar_results.append({
        'fund_id': fund_id,
        'VaR_95': var_95,
        'CVaR_95': cvar_95,
        'VaR_99': var_99,
        'CVaR_99': cvar_99
    })

var_df = pd.DataFrame(var_cvar_results)
var_df.to_csv('../data/processed/adv_var_cvar.csv', index=False)

plt.figure(figsize=(10,6))
sns.barplot(x='fund_id', y='CVaR_95', data=var_df)
plt.title('Conditional VaR (95%) by Fund')
plt.xlabel('Fund ID')
plt.ylabel('Expected Shortfall (CVaR)')
plt.savefig('../charts/adv_cvar_95.png')
plt.close()
var_df


,fund_id,VaR_95,CVaR_95,VaR_99,CVaR_99
0,118632,-0.015859,-0.024661,-0.028168,-0.042334
1,119092,0.000059,-0.000205,-0.000236,-0.000910
2,119551,-0.001019,-0.008438,-0.004134,-0.034298
3,120503,-0.014593,-0.028535,-0.026495,-0.066963
4,120841,-0.014232,-0.023849,-0.027613,-0.042316
5,125497,-0.015102,-0.023230,-0.027222,-0.038803


## 3. Rolling Sharpe Ratio
Analyzing the consistency of risk-adjusted returns over time (252-day window).

In [4]:
window = 252
rf = 0.05 / 252

df_nav['rolling_mean'] = df_nav.groupby('fund_id')['daily_return'].transform(lambda x: x.rolling(window).mean())
df_nav['rolling_std'] = df_nav.groupby('fund_id')['daily_return'].transform(lambda x: x.rolling(window).std())
df_nav['rolling_sharpe'] = (df_nav['rolling_mean'] - rf) / df_nav['rolling_std'] * np.sqrt(252)

plt.figure(figsize=(12,6))
for fund_id, group in df_nav.groupby('fund_id'):
    plt.plot(group['date'], group['rolling_sharpe'], label=str(fund_id))
plt.title('1-Year Rolling Sharpe Ratio')
plt.xlabel('Date')
plt.ylabel('Sharpe Ratio')
plt.legend()
plt.savefig('../charts/adv_rolling_sharpe.png')
plt.close()
print("Rolling Sharpe calculated and charted.")


Rolling Sharpe calculated and charted.


## 4. Investor Cohort Analysis & SIP Continuity
Analyzing investor behavior, cohort retention, and SIP discipline.

In [5]:
# Assuming 'transaction_id' is a proxy for investor_id for this small dataset if investor_id is missing
if 'investor_id' not in df_inv.columns:
    df_inv['investor_id'] = df_inv['transaction_id'] # Proxy

# Cohort Analysis (by acquisition month)
df_inv['cohort_month'] = df_inv.groupby('investor_id')['transaction_date'].transform('min').dt.to_period('M')
df_inv['transaction_month'] = df_inv['transaction_date'].dt.to_period('M')

cohort_data = df_inv.groupby(['cohort_month', 'transaction_month'])['investor_id'].nunique().reset_index()
cohort_data.rename(columns={'investor_id': 'active_investors'}, inplace=True)
cohort_data.to_csv('../data/processed/adv_cohort_analysis.csv', index=False)

# SIP Continuity
sip_data = df_inv[df_inv['transaction_type'].str.upper() == 'SIP'].copy()
sip_counts = sip_data.groupby('investor_id').size().reset_index(name='sip_count')
sip_continuity = sip_counts['sip_count'].value_counts().reset_index()
sip_continuity.columns = ['number_of_sips', 'investor_count']
sip_continuity.to_csv('../data/processed/adv_sip_continuity.csv', index=False)

plt.figure(figsize=(8,5))
sns.barplot(x='number_of_sips', y='investor_count', data=sip_continuity)
plt.title('SIP Continuity Analysis')
plt.xlabel('Number of SIP Installments')
plt.ylabel('Number of Investors')
plt.savefig('../charts/adv_sip_continuity.png')
plt.close()
sip_continuity


,number_of_sips,investor_count
0,1,5


## 5. Sector Concentration Index (HHI)
Herfindahl-Hirschman Index measures portfolio diversification. (Using simulated sector weights if actuals missing)

In [6]:
# Simulating sector weights for funds
np.random.seed(42)
sectors = ['Financials', 'IT', 'Healthcare', 'Consumer', 'Energy', 'Materials']
fund_ids = df_nav['fund_id'].unique()

hhi_results = []
sector_data = []

for fid in fund_ids:
    weights = np.random.dirichlet(np.ones(len(sectors)), size=1)[0]
    # HHI is sum of squared weights * 10000
    hhi = np.sum((weights * 100) ** 2)
    hhi_results.append({'fund_id': fid, 'HHI': hhi})
    
    for s, w in zip(sectors, weights):
        sector_data.append({'fund_id': fid, 'sector': s, 'weight': w})
        
df_hhi = pd.DataFrame(hhi_results)
df_hhi.to_csv('../data/processed/adv_hhi_index.csv', index=False)

plt.figure(figsize=(10,6))
sns.barplot(x='fund_id', y='HHI', data=df_hhi)
plt.title('Sector Concentration Index (HHI)')
plt.axhline(1500, color='orange', linestyle='--', label='Moderate Concentration')
plt.axhline(2500, color='red', linestyle='--', label='High Concentration')
plt.legend()
plt.savefig('../charts/adv_hhi.png')
plt.close()
df_hhi


,fund_id,HHI
0,118632,3254.575597
1,119092,3114.278397
2,119551,3212.523913
3,120503,2135.598055
4,120841,2525.171960
5,125497,2826.105861


## 6. Content-Based Mutual Fund Recommendation System
Recommends similar funds based on risk-return profiles, category, and expenses using Cosine Similarity.

In [7]:
# Extract latest metrics for recommendation
reco_features = []
for fid, group in df_nav.groupby('fund_id'):
    ann_return = group['daily_return'].mean() * 252
    volatility = group['daily_return'].std() * np.sqrt(252)
    reco_features.append({'fund_id': fid, 'return': ann_return, 'risk': volatility, 'sim_expense': np.random.uniform(0.5, 1.5)})

df_features = pd.DataFrame(reco_features).set_index('fund_id')
df_features.replace([np.inf, -np.inf], np.nan, inplace=True)
df_features.dropna(inplace=True)
if not df_features.empty:
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(df_features)

    cos_sim = cosine_similarity(scaled_features)
    df_sim = pd.DataFrame(cos_sim, index=df_features.index, columns=df_features.index)
else:
    df_sim = pd.DataFrame()

# Recommendation function
def recommend_funds(fund_id, top_n=2):
    if fund_id not in df_sim.index:
        return []
    sim_scores = df_sim[fund_id].sort_values(ascending=False)
    # Exclude the fund itself
    sim_scores = sim_scores[sim_scores.index != fund_id]
    return sim_scores.head(top_n).reset_index()

# Test
test_fund = df_features.index[0]
recos = recommend_funds(test_fund)
recos.to_csv('../data/processed/adv_recommendations_sample.csv', index=False)
print(f"Recommendations for {test_fund}:\n", recos)


Recommendations for 118632:
   fund_id    118632
0  120841  0.749274
1  125497  0.525288


## 7. KMeans Investor Segmentation
Clustering investors based on their transaction amount and frequency.

In [8]:
# Aggregate investor behavior
investor_agg = df_inv.groupby('investor_id').agg(
    total_invested=('amount', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()

if len(investor_agg) >= 3:
    scaler_inv = StandardScaler()
    scaled_inv = scaler_inv.fit_transform(investor_agg[['total_invested', 'transaction_count']])
    
    kmeans = KMeans(n_clusters=min(3, len(investor_agg)), random_state=42)
    investor_agg['cluster'] = kmeans.fit_predict(scaled_inv)
    
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=investor_agg, x='transaction_count', y='total_invested', hue='cluster', palette='viridis', s=100)
    plt.title('Investor Segments (KMeans)')
    plt.savefig('../charts/adv_investor_clusters.png')
    plt.close()
    
    investor_agg.to_csv('../data/processed/adv_investor_segments.csv', index=False)
    print("KMeans clustering completed.")
else:
    print("Not enough investors for meaningful clustering.")


KMeans clustering completed.


## 8. Monte Carlo Simulation (5-Year NAV Projection)
Simulating thousands of possible future paths for NAV based on historical drift and volatility.

In [9]:
# Pick one fund for simulation
target_fund = fund_ids[0]
target_data = df_nav[df_nav['fund_id'] == target_fund]['daily_return'].dropna()

mu = target_data.mean()
sigma = target_data.std()
last_nav = df_nav[df_nav['fund_id'] == target_fund]['nav'].iloc[-1]

days = 252 * 5
simulations = 1000

simulated_paths = np.zeros((days, simulations))
simulated_paths[0] = last_nav

for t in range(1, days):
    shock = np.random.normal(mu, sigma, simulations)
    simulated_paths[t] = simulated_paths[t-1] * (1 + shock)

plt.figure(figsize=(10,6))
plt.plot(simulated_paths[:, :100], color='blue', alpha=0.05) # Plot 100 paths
plt.title(f'Monte Carlo Simulation (5-Year NAV) - Fund {target_fund}')
plt.xlabel('Trading Days')
plt.ylabel('Projected NAV')
plt.savefig('../charts/adv_monte_carlo.png')
plt.close()
print(f"Monte Carlo median end NAV: {np.median(simulated_paths[-1]):.2f}")


Monte Carlo median end NAV: 205.17


## 9. Markowitz Efficient Frontier
Finding the optimal portfolio weights for maximum Sharpe Ratio.

In [10]:
# Pivot returns
returns_pivot = df_nav.pivot(index='date', columns='fund_id', values='daily_return').dropna()

if returns_pivot.shape[1] > 1:
    mean_returns = returns_pivot.mean() * 252
    cov_matrix = returns_pivot.cov() * 252
    num_assets = len(mean_returns)
    
    def portfolio_performance(weights, mean_returns, cov_matrix):
        returns = np.sum(mean_returns * weights)
        std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return std, returns
        
    def neg_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate):
        p_std, p_ret = portfolio_performance(weights, mean_returns, cov_matrix)
        return -(p_ret - risk_free_rate) / p_std
        
    args = (mean_returns, cov_matrix, 0.05)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    init_guess = num_assets * [1. / num_assets]
    
    opt_results = sco.minimize(neg_sharpe_ratio, init_guess, args=args, method='SLSQP', bounds=bounds, constraints=constraints)
    
    opt_weights = opt_results.x
    opt_df = pd.DataFrame({'fund_id': mean_returns.index, 'optimal_weight': opt_weights})
    opt_df.to_csv('../data/processed/adv_optimal_weights.csv', index=False)
    
    plt.figure(figsize=(8,6))
    sns.barplot(x='fund_id', y='optimal_weight', data=opt_df)
    plt.title('Optimal Portfolio Weights (Max Sharpe)')
    plt.savefig('../charts/adv_optimal_weights.png')
    plt.close()
    print("Efficient Frontier optimized.")


Efficient Frontier optimized.


## 10. ARIMA NAV Forecasting
Using AutoRegressive Integrated Moving Average (ARIMA) to forecast short-term NAV.

In [11]:
# Forecast for one fund
target_data = df_nav[df_nav['fund_id'] == target_fund][['date', 'nav']].set_index('date')
# Resample to weekly to smooth noise for ARIMA
weekly_nav = target_data['nav'].resample('W').last().dropna()

# Fit ARIMA(5,1,0) as a basic baseline model
model = ARIMA(weekly_nav, order=(5, 1, 0))
model_fit = model.fit()

forecast_steps = 12 # 12 weeks
forecast = model_fit.forecast(steps=forecast_steps)
forecast.index = pd.date_range(start=weekly_nav.index[-1] + pd.Timedelta(weeks=1), periods=forecast_steps, freq='W')

plt.figure(figsize=(10,6))
plt.plot(weekly_nav.index[-50:], weekly_nav.iloc[-50:], label='Historical NAV (Weekly)')
plt.plot(forecast.index, forecast, color='red', label='ARIMA Forecast')
plt.title(f'ARIMA NAV Forecast (12 Weeks) - Fund {target_fund}')
plt.xlabel('Date')
plt.ylabel('NAV')
plt.legend()
plt.savefig('../charts/adv_arima_forecast.png')
plt.close()

forecast.to_csv('../data/processed/adv_arima_forecast.csv')
print("ARIMA Forecasting completed.")


ARIMA Forecasting completed.
